In [ ]:
# imports
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import insightface
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image
import os
import glob
import ipywidgets
from tqdm import tqdm


In [ ]:
# load LFW all names
path_to_csv = "/home/sho/Insightface-Face-Recognition/img/LFW/lfw_allnames.csv"
path_to_img = "/home/sho/Insightface-Face-Recognition/img/LFW/lfw-deepfunneled/lfw-deepfunneled/"

df = pd.read_csv(path_to_csv)
print(df.shape)
df.head()


(5749, 2)


,name,images
0,AJ_Cook,1
1,AJ_Lamas,1
2,Aaron_Eckhart,1
3,Aaron_Guiel,1
4,Aaron_Patterson,1


In [ ]:
# add an ID column for each person
df['ID'] = range(len(df))

# add a column for each person images path
df['name_path'] = path_to_img + df['name']

df.head()

,name,images,ID,name_path
0,AJ_Cook,1,0,/home/sho/Insightface-Face-Recognition/img/LFW...
1,AJ_Lamas,1,1,/home/sho/Insightface-Face-Recognition/img/LFW...
2,Aaron_Eckhart,1,2,/home/sho/Insightface-Face-Recognition/img/LFW...
3,Aaron_Guiel,1,3,/home/sho/Insightface-Face-Recognition/img/LFW...
4,Aaron_Patterson,1,4,/home/sho/Insightface-Face-Recognition/img/LFW...


In [ ]:
# img to embedding

def img_to_embedding(img_path):

    # read img
    read_img = cv2.imread(img_path)

    # init face analysis app
    app = FaceAnalysis()

    # to show images
    """plt.imshow(read_img[:,:,::-1])
    plt.show()"""

    # to prepare images 
    app.prepare(ctx_id=0)
    try:
        faces = app.get(read_img)[0]['embedding']

        return faces
    
    except Exception as e:
        print(f"Error in embedding function for {img_path}: {e}")
        return None

    


In [5]:
test = '/home/sho/Insightface-Face-Recognition/img/LFW/lfw-deepfunneled/lfw-deepfunneled/Alimzhan_Tokhtakhounov/Alimzhan_Tokhtakhounov_0001.jpg'
img_to_embedding(test)




/home/sho/anaconda3/envs/face-gen-notebook/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (64

array([ 3.03388268e-01, -1.16919637e+00,  9.98994231e-01,  1.16959047e+00,
       -5.30745149e-01,  1.49588096e+00, -4.98472273e-01,  9.64588344e-01,
       -1.49702832e-01, -1.54247060e-01, -5.37674785e-01,  4.34018224e-01,
        3.37939799e-01, -8.90786648e-01,  3.68266612e-01, -1.61088562e+00,
       -7.11225569e-01, -6.69753432e-01, -6.14364326e-01, -6.71382189e-01,
        1.20678830e+00, -3.47654745e-02, -3.78919691e-02, -9.20734882e-01,
       -4.56830561e-02, -9.57407236e-01, -1.11521721e+00,  1.70577931e+00,
        1.09602904e+00,  7.83460259e-01, -5.41847527e-01,  3.82343620e-01,
       -6.57285810e-01,  1.06513333e+00, -1.59068525e+00,  3.75739098e-01,
       -1.86584139e+00, -1.42555952e+00,  6.11202002e-01,  5.21783769e-01,
        3.44935954e-01, -8.31741929e-01,  1.20857105e-01, -2.67411679e-01,
       -5.15638232e-01, -2.47151747e-01,  1.38737285e+00,  3.29087138e-01,
        7.59184062e-01, -1.71835339e+00,  1.07944459e-02, -5.17993569e-01,
       -9.89799142e-01,  

In [16]:
from tqdm.auto import tqdm
import contextlib
import io
import os
import glob
import pandas as pd

embeddings_data = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Creating embeddings...", dynamic_ncols=True):
    person_name = row['name']
    person_id = row['ID']
    person_path = row['name_path']

    image_paths = glob.glob(os.path.join(person_path, '*.jpg'))

    for img_path in image_paths:
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                embedding = img_to_embedding(img_path)

            if embedding is None:
                continue

            embeddings_data.append({
                'name': person_name,
                'ID': person_id,
                'embedding': embedding,
                'embedding type': type(embedding),
                'embedding dtype': embedding.dtype,
                'embedding_size': embedding.size,
                'image_name': os.path.basename(img_path),
                'filepath': img_path
            })

        except Exception as e:
            tqdm.write(f"❌ Error on image: {img_path} | {e}")

/home/sho/anaconda3/envs/face-gen-notebook/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating embeddings...: 100%|██████████| 5749/5749 [5:18:25<00:00,  3.32s/it]     


In [ ]:
"""embeddings_data = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="creating embeddings..."):
    person_name = row['name']
    person_id = row['ID']
    person_path = row['name_path']
    
    # Find all jpg images in this person's folder
    image_paths = glob.glob(os.path.join(person_path, '*.jpg'))
    
    for img_path in image_paths:
        embedding = img_to_embedding(img_path)
        if embedding is None:
            continue
        
        embeddings_data.append({
            'name': person_name,
            'ID': person_id,
            'embedding': embedding, 
            'embedding type': type(embedding),
            'embedding dtype': embedding.dtype,
            'embedding_size': embedding.size, 
            'image_name': os.path.basename(img_path),
            'filepath': img_path
        })"""

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [14]:
df_embeddings = pd.DataFrame(embeddings_data)
df_embeddings.shape
df_embeddings.head()

,name,ID,embedding,embedding type,embedding dtype,embedding_size,image_name,filepath
0,AJ_Cook,0,"[-0.3419863, 0.6712041, -1.2866051, 0.60350394...",<class 'numpy.ndarray'>,float32,512,AJ_Cook_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
1,AJ_Lamas,1,"[0.059667192, 0.076293096, 1.3340704, 1.245711...",<class 'numpy.ndarray'>,float32,512,AJ_Lamas_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
2,Aaron_Eckhart,2,"[1.2447274, -0.9747005, 1.2987878, 0.46920043,...",<class 'numpy.ndarray'>,float32,512,Aaron_Eckhart_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
3,Aaron_Guiel,3,"[-0.17151171, -1.2514898, -0.38039768, -0.0818...",<class 'numpy.ndarray'>,float32,512,Aaron_Guiel_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
4,Aaron_Patterson,4,"[0.41729146, -0.24729499, -0.86146384, -2.1820...",<class 'numpy.ndarray'>,float32,512,Aaron_Patterson_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...


In [20]:
df_embeddings = pd.DataFrame(embeddings_data)
print(df_embeddings.shape)
df_embeddings.head()
df_embeddings.to_csv("./embeddings.csv", index=False)

(13195, 8)


In [2]:
import pandas as pd

In [4]:
# reload embeddings df
df_emb = pd.read_csv("./embeddings.csv")
df_emb.shape

(13195, 8)

In [4]:
df_emb.head()

,name,ID,embedding,embedding type,embedding dtype,embedding_size,image_name,filepath
0,AJ_Cook,0,[-3.41986299e-01 6.71204090e-01 -1.28660512e+...,<class 'numpy.ndarray'>,float32,512,AJ_Cook_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
1,AJ_Lamas,1,[ 5.96671924e-02 7.62930959e-02 1.33407044e+...,<class 'numpy.ndarray'>,float32,512,AJ_Lamas_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
2,Aaron_Eckhart,2,[ 1.24472737e+00 -9.74700511e-01 1.29878783e+...,<class 'numpy.ndarray'>,float32,512,Aaron_Eckhart_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
3,Aaron_Guiel,3,[-0.17151171 -1.2514898 -0.38039768 -0.081882...,<class 'numpy.ndarray'>,float32,512,Aaron_Guiel_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
4,Aaron_Patterson,4,[ 0.41729146 -0.24729499 -0.86146384 -2.182007...,<class 'numpy.ndarray'>,float32,512,Aaron_Patterson_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...


In [10]:
type(df_emb.embedding[0])

str

In [11]:
df_emb.embedding[0]

'[-3.41986299e-01  6.71204090e-01 -1.28660512e+00  6.03503942e-01\n -3.18364918e-01 -4.68093991e-01 -2.34770045e-01  3.22836861e-02\n  7.13358521e-01 -6.83606386e-01 -6.01341248e-01 -2.26560807e+00\n  1.31078124e+00  1.09312689e+00  5.34345269e-01  1.09747398e+00\n -4.83847260e-01 -2.11912215e-01 -2.36376584e-01 -3.22396383e-02\n  2.48618588e-01 -1.68814075e+00 -1.91167235e+00  4.39560950e-01\n  4.66680527e-01 -5.63056290e-01  8.96633506e-01  1.81865704e+00\n  6.78355932e-01 -1.46311998e+00 -1.45712420e-01 -7.37816811e-01\n  9.33966041e-01  4.59790051e-01  5.24055779e-01 -3.94828200e-01\n -1.12502050e+00 -7.19395816e-01 -9.17202830e-01 -4.87413019e-01\n  1.40539682e+00  1.25581741e-01 -8.22785020e-01 -7.52641618e-01\n  5.46758711e-01 -6.08044565e-01  2.52439946e-01  4.20660019e-01\n  6.25930011e-01  5.21705449e-01 -1.10720861e+00  7.13488579e-01\n -8.57424736e-01  6.78034306e-01  9.67450202e-01  1.18356633e+00\n  5.84251583e-01  6.37974858e-01  1.24099672e-01 -1.52626216e+00\n  1.64706

In [14]:
# convert csv to pkl
df = pd.read_csv("./embeddings.csv")

import ast
import numpy as np

df['embedding'] = df['embedding'].apply(
    lambda x: np.fromstring(x.strip('[]'), sep=' ')
)
#df['embedding'] = df['embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

# Save it as a Pickle file
df.to_pickle("./embeddings-0312.pkl")

# start here

In [1]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import insightface

In [2]:
df = pd.read_pickle("./embeddings-0312.pkl")

print(type(df['embedding'].iloc[0]))  # <class 'numpy.ndarray'>
is_array_col = df['embedding'].apply(lambda x: isinstance(x, np.ndarray))
print(is_array_col.value_counts())

# convert back to float32
df['embedding'] = df['embedding'].apply(lambda x: x.astype(np.float32))

<class 'numpy.ndarray'>
embedding
True    13195
Name: count, dtype: int64


In [3]:
df['embedding'].iloc[0].dtype

dtype('float32')

# sanity check

In [ ]:
import random

detector = insightface.model_zoo.get_model('/home/sho/.insightface/models/buffalo_l/w600k_r50.onnx')
#app = FaceAnalysis()

# Select a random embedding
query_idx = random.randint(0, len(df) - 1)
query_emb = df['embedding'].iloc[query_idx]

similarities = []

for i in tqdm(range(len(df)), desc="Searching similar embeddings", dynamic_ncols=True):
    emb = df['embedding'].iloc[i]
    sim = detector.compute_sim(query_emb, emb)
    #print(f'Similarity score: {sim}')
    similarities.append(sim)

# Find top match
top_match_idx = np.argmax(similarities)

print(f"Query index: {query_idx}")
print(f"Top match index: {top_match_idx}")
print(f"Similarity score: {similarities[top_match_idx]:.6f}")




/home/sho/anaconda3/envs/face-gen-notebook/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


Searching similar embeddings:   0%|          | 0/13195 [00:00<?, ?it/s]

Similarity score: -0.07656122744083405
Similarity score: -0.0196909811347723
Similarity score: 0.10059366375207901
Similarity score: -0.045011185109615326
Similarity score: -0.08891334384679794
Similarity score: -0.008606958203017712
Similarity score: -0.037452057003974915
Similarity score: -0.05270988121628761
Similarity score: 0.00015827461902517825
Similarity score: -0.08940943330526352
Similarity score: -0.0745629221200943
Similarity score: -0.05183979123830795
Similarity score: -0.0893077403306961
Similarity score: 0.10029501467943192
Similarity score: -0.02859790436923504
Similarity score: 0.0018946267664432526
Similarity score: -0.07020122557878494
Similarity score: -0.03331659734249115
Similarity score: -0.07494957745075226
Similarity score: -0.005163185298442841
Similarity score: -0.033646076917648315
Similarity score: -0.028748519718647003
Similarity score: 0.014923726208508015
Similarity score: -0.05421985313296318
Similarity score: -0.027627335861325264
Similarity score: -0

In [ ]:
1

In [ ]:
len(similarities)

In [ ]:
similarities#.sort(key=lambda x: x[1], reverse=True)

: 

In [ ]:
# Step 3: Sort by similarity score (descending)
similarities.sort(key=lambda x: x[1], reverse=True)

# Step 4: Get top 3 matches
top_3 = similarities[:3]

# Step 5: Show results
print(f"\nQuery index: {query_idx} | Query name: {df.iloc[query_idx]['name']}")
print("\nTop 3 similar embeddings:")
for rank, (idx, score) in enumerate(top_3, 1):
    print(f"Rank {rank}: Index={idx}, Name={df.iloc[idx]['name']}, Similarity Score={score:.4f}")

# Optional: return values programmatically
top_3_indices = [idx for idx, _ in top_3]
top_3_scores = [score for _, score in top_3]

: 

In [ ]:
1+1